In [3]:
# Import csv file of hub deliveries for data exploration

import pandas as pd

df = pd.read_csv('/Users/DianaLara/Downloads/Orders.csv', encoding='UTF-16', sep='\t')
print(df.shape)
df.head()


(27979, 15)


,Order ID,Actual Delivery Date,Delay Reason,Driver ID,Driver Name,Hub Name,Is Delayed,Is On Time,Order Date,Order Status,Vehicle Name,Vehicle Type,Customer Satisfaction Score,Delivery Time Hours,Hub Processing Time Hours
0,1,25-10-2024,NaN,43,Karen Rodriguez,San Antonio Hub,False,True,25-10-2024,Delivered,FT-036,Truck,4,6.81,0.89
1,2,16-06-2024,NaN,29,Matthew Williams,Houston Hub,False,True,16-06-2024,Delivered,FT-016,Van,4,5.74,3.60
2,3,05-07-2024,NaN,25,Nancy Harris,Austin Hub,False,True,05-07-2024,Delivered,FT-040,Van,4,12.91,2.07
3,4,22-08-2023,NaN,20,David Davis,Fort Worth Hub,False,True,22-08-2023,Delivered,FT-039,Van,5,9.40,2.37
4,5,06-06-2024,Severe Weather,49,Joseph Williams,Dallas Main Hub,True,False,02-06-2024,Delivered,FT-018,Truck,3,103.48,1.80


In [4]:
# Columns

print(len(df.columns))
print(list(df.columns))

15
['Order ID', 'Actual Delivery Date', 'Delay Reason', 'Driver ID', 'Driver Name', 'Hub Name', 'Is Delayed', 'Is On Time', 'Order Date', 'Order Status', 'Vehicle Name', 'Vehicle Type', 'Customer Satisfaction Score', 'Delivery Time Hours', 'Hub Processing Time Hours']


In [ ]:
# Extracting the data type from the Delivery Time Hours column

print(df['Delivery Time Hours'].dtype)

In [18]:
# Total delivery hours

total_hours = sum(df['Delivery Time Hours'])
print(f"Total delivery hours across all orders: {total_hours}")

Total delivery hours across all orders: nan


In [19]:
# Total delivery hours calculation, skipping null values

print(df['Delivery Time Hours'].isna().sum())

252


In [20]:
# Percentage of rows in Delivery Time Hours missing values

print(f"{252/len(df)*100:.2f}% of rows are missing Delivery Time Hours")

0.90% of rows are missing Delivery Time Hours


In [21]:
# Extracting Order Status to figure out if that has to do with missing values in Delivery Tinme Hours

print(df[df['Delivery Time Hours'].isna()]['Order Status'].value_counts())

Order Status
Cancelled    252
Name: count, dtype: int64


In [7]:
# Average delivery time

avg_delivery_time = df['Delivery Time Hours'].mean()
print(f"Average Delivery Time: {avg_delivery_time:.2f} hours")

Average Delivery Time: 35.78 hours


In [8]:
# Fastest delivery time

fastest_delivery = df['Delivery Time Hours'].min()
print(f"Fastest Delivery Time: {fastest_delivery:.2f} hours")

Fastest Delivery Time: 2.27 hours


In [9]:
# Top ten fastest delivery orders

fastest_order = df.loc[df['Delivery Time Hours'].idxmin()]
print(fastest_order)

Order ID                                 4271
Actual Delivery Date               08-09-2023
Delay Reason                              NaN
Driver ID                                   2
Driver Name                    Jennifer Lopez
Hub Name                          Houston Hub
Is Delayed                              False
Is On Time                               True
Order Date                         08-09-2023
Order Status                        Delivered
Vehicle Name                           FT-003
Vehicle Type                              Van
Customer Satisfaction Score                 5
Delivery Time Hours                      2.27
Hub Processing Time Hours                1.67
Name: 26132, dtype: object


In [10]:
# Top ten fastest deliveries

df['Delivery Time Hours'].sort_values().head(10)

26132    2.27
12614    2.31
14611    2.33
13981    2.33
14319    2.33
10917    2.34
27857    2.34
22039    2.36
12599    2.36
16255    2.38
Name: Delivery Time Hours, dtype: float64

In [ ]:
# Slowest delivery time

slowest_delivery = df['Delivery Time Hours'].max()
print(f"Slowest Delivery Time: {slowest_delivery:.2f} hours")

In [11]:
# Slowest delivery

slowest_delivery = df.loc[df['Delivery Time Hours'].idxmax()]
print(slowest_delivery)

Order ID                                       26190
Actual Delivery Date                      06-04-2024
Delay Reason                   Package Sorting Error
Driver ID                                         11
Driver Name                             Linda Taylor
Hub Name                             San Antonio Hub
Is Delayed                                      True
Is On Time                                     False
Order Date                                01-04-2024
Order Status                               Delivered
Vehicle Name                                  FT-036
Vehicle Type                                   Truck
Customer Satisfaction Score                        4
Delivery Time Hours                           143.14
Hub Processing Time Hours                       1.74
Name: 3238, dtype: object


In [16]:
# Number of Hubs

print(len(df))  # total number of rows/orders
print(len(df['Hub Name'].unique()))  # number of unique hubs

27979
6


In [12]:
# Hubs with the slowest deliveries

df.groupby('Hub Name')['Delivery Time Hours'].mean().sort_values(ascending=False)

Hub Name
Fort Worth Hub     36.500841
San Antonio Hub    35.791908
Houston Hub        35.789620
Dallas Main Hub    35.783730
Austin Hub         35.532736
El Paso Hub        35.238369
Name: Delivery Time Hours, dtype: float64

In [13]:
# Percentage of deliveries made under 3 days

under_3_days = df[df['Delivery Time Hours'] <= 72]
pct_under_3_days = (len(under_3_days) / len(df)) * 100
print(f"Percentage under 3 days: {pct_under_3_days:.2f}%")

Percentage under 3 days: 92.74%


In [15]:
# Delay reason breakdown

def delay_reason_breakdown(df, hours=None, comparison='over'):
    """Show delay reason counts (and %) for deliveries matching a threshold.
    If hours=None, shows delay reasons for all deliveries."""
    if hours is not None:
        if comparison == 'over':
            subset = df[df['Delivery Time Hours'] > hours]
        else:
            subset = df[df['Delivery Time Hours'] < hours]
        print(f"\nDelay reasons for deliveries {comparison} {hours} hours (n={len(subset)}):")
    else:
        subset = df
        print(f"\nDelay reasons for all deliveries (n={len(subset)}):")

    counts = subset['Delay Reason'].value_counts()
    pct = subset['Delay Reason'].value_counts(normalize=True) * 100

    summary = pd.DataFrame({'count': counts, 'pct': pct.round(2)})
    return summary

delay_reason_breakdown(df, 120, 'over')


Delay reasons for deliveries over 120 hours (n=168):


,count,pct
Delay Reason,,
Driver Unavailable,24,14.29
Incorrect Address,21,12.50
Vehicle Breakdown,20,11.90
Hub Processing Delay,19,11.31
Multiple Delivery Stops,17,10.12
Road Construction,16,9.52
Severe Weather,16,9.52
Package Sorting Error,13,7.74
Customer Not Home,11,6.55
